<a href="https://colab.research.google.com/github/mythien107/busad878/blob/main/Bullwhip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# 1. Set up the timeline (40 weeks)
weeks = np.arange(1, 41)

# Helper function to generate the delayed waves
def create_wave(weeks, start_week, peak_week, base_demand, amplitude):
    wave = np.zeros(len(weeks))
    for i, w in enumerate(weeks):
        if w >= start_week:
            dist = w - peak_week
            wave[i] = amplitude * np.exp(-0.10 * (dist ** 2))

    dip = np.zeros(len(weeks))
    for i, w in enumerate(weeks):
        if w > peak_week + 4:
            dip[i] = -(amplitude * 0.3) * np.exp(-0.15 * ((w - (peak_week + 6)) ** 2))

    demand_line = np.where(weeks < 5, 4, base_demand)
    return demand_line + wave + dip

# 2. The main plotting function that updates when sliders move
def plot_bullwhip(jump, panic):
    base_demand = 4 + jump

    # Recalculate how the panic compounds up the chain
    a_r = jump * panic
    a_w = a_r * panic
    a_d = a_w * panic
    a_f = a_d * panic

    # Generate the data
    customer_demand = np.where(weeks < 5, 4, base_demand)
    retailer = create_wave(weeks, 5, 8, base_demand, a_r)
    wholesaler = create_wave(weeks, 6, 12, base_demand, a_w)
    distributor = create_wave(weeks, 8, 16, base_demand, a_d)
    factory = create_wave(weeks, 10, 22, base_demand, a_f)

    # Clear previous plots and set up the canvas
    plt.figure(figsize=(12, 7))

    # Plot the lines
    plt.plot(weeks, customer_demand, label='Customer Demand', linestyle='--', color='black', linewidth=2)
    plt.plot(weeks, retailer, label='Retailer', color='#2ca02c', linewidth=2.5)
    plt.plot(weeks, wholesaler, label='Wholesaler', color='#1f77b4', linewidth=2.5)
    plt.plot(weeks, distributor, label='Distributor', color='#ff7f0e', linewidth=2.5)
    plt.plot(weeks, factory, label='Factory', color='#d62728', linewidth=2.5)

    # Formatting
    plt.title('The Bullwhip Effect: Interactive Simulation', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Weeks', fontsize=12, fontweight='bold')
    plt.ylabel('Order Quantity (Cases)', fontsize=12, fontweight='bold')
    plt.xlim(1, 40)

    # Dynamically adjust the Y-axis so the red factory line never goes off-screen
    max_y = max(50, np.max(factory) * 1.1)
    plt.ylim(0, max_y)

    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

# 3. Create the interactive widgets and link them to the plotting function
widgets.interact(plot_bullwhip,
                 jump=widgets.FloatSlider(value=4.0, min=0.0, max=20.0, step=1.0, description='Demand Jump:'),
                 panic=widgets.FloatSlider(value=1.5, min=1.1, max=2.5, step=0.1, description='Panic Factor:'))

interactive(children=(FloatSlider(value=4.0, description='Demand Jump:', max=20.0, step=1.0), FloatSlider(valu…

<function __main__.plot_bullwhip(jump, panic)>

<details>

Note that Customer Demand (the black dotted line) is flat at 4 cases for the first 4 weeks, then permanently steps up by the value set by 'Demand Jump' slider. In the real world, this is a minor market shift. Maybe a new marketing campaign worked, or a competitor went out of business. It is a stable, predictable, and manageable increase.

Between Weeks 5 and 15, the green (Retailer) and blue (Wholesaler) lines begin to rise above the new baseline of Customer Demand. The orange (Distributor) and red (Factory) lines shoot up exponentially, often reaching 40, 60, or 80+ units. Because they cannot see the true customer demand, they interpret their empty shelves as a massive, ongoing boom in popularity:

*   The Retailer orders extra to cover the new demand plus their growing backlog.
*   The Wholesaler sees this double-order and panics, doubling it again to be safe.

By the time the signal reaches the Factory, Manufacturers are convinced the product is a global sensation and ramp up production to maximum capacity. This is the "crack of the whip."

From Weeks 16-25, the massive peaks suddenly crash. The lines drop rapidly, often plummeting completely to zero. The delayed shipments finally start arriving. Suddenly, the Wholesaler and Distributor receive those massive orders of 40 or 80 cases they placed weeks ago. Because actual customer demand is only x cases a week, their warehouses instantly overflow. They realize their mistake and stop ordering entirely to try and bleed off their excess stock.

Starting from Week 26, the lines eventually flatten out and merge with the black customer demand line. The panic has passed, the excess inventory has been sold off, and the supply chain has finally stabilized at the correct new demand level. However, the financial damage (the costs of massive backlogs followed by massive warehousing fees) is already done.

</details>